In [ ]:
# 快为第一准则，先把模型跑起来
using Revise
using BEPS, RTableTools, DataFrames, Dates, ModelParams
using Ipaper


function agg_daily(df_fluxes, dates)
  dates_day = Date.(dates)

  GPP_sim = df_fluxes.GPP
  ET_sim = df_fluxes.Trans + df_fluxes.Evap
  fun = nansum
  (;
    GPP_sim=apply(GPP_sim, 1; by=dates_day, fun),
    ET_sim=apply(ET_sim, 1; by=dates_day, fun),
    dates_day=unique_sort(dates_day)
  )
end

parse_time(x::AbstractString) = DateTime(x, dateformat"yyyy-mm-ddTHH:MM:SSZ")


parse_time (generic function with 1 method)

In [ ]:
indir = "Z:/GitHub/jl-pkgs/ChinaFlux2026/data-raw/BEPS"
indir = "/mnt/z/GitHub/jl-pkgs/ChinaFlux2026/data-raw/BEPS"

@time FORCING = fread("$indir/Forcing_Met_Hourly_BEPS_Forest_sp12_hourly_v20260507.csv")
replace_missing!(FORCING)
SITES = unique(FORCING.site)

@time FluxLAI = fread("$indir/Forcing_FluxLAI_Daily_BEPS_Forest_sp12_v20260506.csv")
replace_missing!(FluxLAI)
FluxLAI[1:6, :]

  2.442718 seconds (9.13 M allocations: 616.168 MiB, 7.15% gc time, 50 lock conflicts, 758.42% compilation time: 33% of which was recompilation)
  0.321702 seconds (938.74 k allocations: 53.698 MiB, 7.17% gc time, 50 lock conflicts, 455.48% compilation time)


Row,site,date,name,NEE,RE,GPP,ET,Rn,Rs,LE,Hs,VPD,VPD_canopy,SM1,LAI_glass_G005,LAI_glass,LAI_whit
,String,Date,String31,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,DBF_天然栎林_宝天曼,2017-01-01,宝天曼,-0.231944,0.132241,0.364185,-0.324487,6.52229,20.4633,-9.36855,-16.8075,0.181504,0.220053,0.196877,0.5,0.5,0.3
2,DBF_天然栎林_宝天曼,2017-01-02,宝天曼,-0.261084,0.155188,0.416272,0.468353,46.7202,128.034,13.489,34.7684,0.445068,0.503705,0.195923,0.5,0.5,0.3
3,DBF_天然栎林_宝天曼,2017-01-03,宝天曼,-0.246388,0.165811,0.412199,0.528722,42.5667,120.42,15.2136,34.6958,0.42824,0.51258,0.1959,0.5,0.5,0.3
4,DBF_天然栎林_宝天曼,2017-01-04,宝天曼,-0.0730873,0.298884,0.371971,-0.39646,4.30708,23.0896,-11.4112,-7.96622,0.147109,0.183145,0.196196,0.5,0.5,0.3
5,DBF_天然栎林_宝天曼,2017-01-05,宝天曼,-0.0221345,0.386657,0.408791,0.371058,24.2848,30.696,10.709,-8.82575,0.0,0.0,0.197683,0.5,0.5,0.3
6,DBF_天然栎林_宝天曼,2017-01-06,宝天曼,0.0720972,0.467639,0.395542,0.462702,19.9706,25.7606,13.3719,-18.4237,0.0,0.000592833,0.20024,0.5,0.5,0.3


In [ ]:
SITE = SITES[8]
d_forcing = FORCING[FORCING.site.==SITE, 2:end]
rename!(d_forcing, [:Ta_canopy => :Tair, :RH_canopy => :RH, :WS_canopy => :Uz])
(; Tair, RH, Uz, Rs, Rln_in, Prcp) = d_forcing
# Prcp = Prcp ./ 1000.0  # 转换为[mm] -> [m]

dates = parse_time.(d_forcing.time)
dates_model = dates .- Hour(8) # [local] -> [UTC]
ntime = length(dates)

forcing = MetSeries(; ntime, Rs, Rln_in, Tair, RH, Uz, Prcp)

d_flux = FluxLAI[FluxLAI.site.==SITE, [:GPP, :ET, :LAI_glass_G005]]
rename!(d_flux, :LAI_glass_G005 => :lai, :GPP => :GPP_obs, :ET => :ET_obs)
(; lai, GPP_obs, ET_obs) = d_flux

GPP_obs = -GPP_obs # gC m-2 day-1, [GEE] -> [GPP]
ET_obs = ET_obs * 1e+06 / 86400; # [MJ m-2] -> [W m-2], ET的最终单位是mm


In [ ]:
VegType::Int = 1
SoilType::Int = 8
model = ParamBEPS(VegType, SoilType)
model.veg.z_wind = 39.6
model.veg.VCmax25 = 56.25
model.veg.g1_w = 4.8
clumping = 0.58

Ta = forcing.Tair[1]
Tsoil0 = Ta
θ0 = model.hydraulic.θ_vfc[1]  # 初始化为田间持水量（避免低于凋萎含水量）
z_snow0 = 0.0
state, model = setup(model; Ta, Tsoil=Float64(Tsoil0), θ0=Float64(θ0), z_snow=Float64(z_snow0))
state


VARS_STATE = [:z_water, :ρ_snow, :z_snow, :r_rain_g, :f_soilwater, :θ, :Tsoil_c, 
	:ETi, :r_waterflow, :G]
VARS_CACHE = [:Ci_new, :Cs_new, :Tc_new, :Gs_new, :Gc, :Gh, :Gw, :Gw_wet]

@time df_fluxes, df_ET, states, caches = simulate(forcing, lai, dates_model; 
	ps=model, state,
    VARS_STATE=VARS_STATE, VARS_CACHE=VARS_CACHE,
    lon=115.06, lat=26.74, clumping);
"ok"

  1.796655 seconds (7.41 M allocations: 378.213 MiB, 3.25% gc time)


"ok"

In [ ]:
(GPP_sim, ET_sim, dates_day) = agg_daily(df_fluxes, dates)
DataFrame([
    (; var="GPP", GOF(GPP_obs, GPP_sim)...),
    (; var="ET", GOF(ET_obs, ET_sim)...)
])

Row,var,NSE,R2,KGE,R,RMSE,MAE,bias,bias_perc,n_valid
,String,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Int64
1,GPP,0.840096,0.847022,0.920257,0.920338,1.08983,0.810039,-0.00259933,-0.0517471,4748
2,ET,0.78563,0.83084,0.84834,0.911504,0.643864,0.487553,-0.120165,-6.30873,4748


In [ ]:
length(dates) / (24 * 365)

13.008219178082191

In [ ]:
states

────────────────────────────────────────────────────
├─ z_water      Vector{Float64} | (113952,) | 0.87 Mb
├─ ρ_snow       Vector{Float64} | (113952,) | 0.87 Mb
├─ z_snow       Vector{Float64} | (113952,) | 0.87 Mb
├─ r_rain_g     Vector{Float64} | (113952,) | 0.87 Mb
└─ f_soilwater  Vector{Float64} | (113952,) | 0.87 Mb
────────────────────────────────────────────────────
├─ θ            Matrix{Float64} | (5, 113952) | 4.35 Mb
├─ Tsoil_c      Matrix{Float64} | (5, 113952) | 4.35 Mb
├─ ETi          Matrix{Float64} | (5, 113952) | 4.35 Mb
├─ r_waterflow  Matrix{Float64} | (5, 113952) | 4.35 Mb
└─ G            Matrix{Float64} | (5, 113952) | 4.35 Mb
────────────────────────────────────────────────────

In [ ]:
Dict(caches)

├─ Ci_new  Matrix{Float64} | (113952, 4) | 3.48 Mb
├─ Cs_new  Matrix{Float64} | (113952, 4) | 3.48 Mb
├─ Tc_new  Matrix{Float64} | (113952, 4) | 3.48 Mb
├─ Gs_new  Matrix{Float64} | (113952, 4) | 3.48 Mb
├─ Gc      Matrix{Float64} | (113952, 4) | 3.48 Mb
├─ Gh      Matrix{Float64} | (113952, 4) | 3.48 Mb
├─ Gw      Matrix{Float64} | (113952, 4) | 3.48 Mb
└─ Gw_wet  Matrix{Float64} | (113952, 4) | 3.48 Mb


## 绘图检查